# JiT-S2-VMamba \u2014 SSC-abc WITHOUT adaLN (literal DiM-2 + A-gate)\n\nAs the bc-noadaln arm, plus the per-direction decay gate `s = \u03c3(\u27e8w_A, z\u27e9 + g\u2080)`. With adaLN present, the gate was shown to work exactly as DiM-2 hypothesised \u2014 s falls with noise level, `range_t` \u2248 10\u00d7 `range_y` \u2014 yet bought nothing (abc \u2248 bc at every diagnostic, 94.85 vs 94.37 FID). Here the gate is one of the *only* conditioning channels, so it is a fairer test of whether timestep-dependent memory decay can carry conditioning on its own.\n\n**31.64M \u2192 21.03M params (\u221233.6%)**. Note abc still pays ~22% throughput for the out-of-kernel fp32 softplus, now the dominant cost.\n\n---\n\nJiT-S2-VMamba on Tiny-ImageNet-200 (64px, patch 8, 200 classes) via the repo's `run_experiment.py` / `evaluate.py`. Comment out the download cell if you attach a dataset instead, and the eval cell if you only want to train.

## 1. Environment  *(Internet ON)*

In [ ]:
import os

# 1) Pin torch to 2.5.1
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 \
    --index-url https://download.pytorch.org/whl/cu124

# 2) Download wheels with explicit destination
CAUSAL = "causal_conv1d-1.5.0.post8+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"
MAMBA  = "mamba_ssm-2.2.4+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"

os.system(f"wget -q https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/{CAUSAL} -O /kaggle/working/{CAUSAL}")
os.system(f"wget -q https://github.com/state-spaces/mamba/releases/download/v2.2.4/{MAMBA} -O /kaggle/working/{MAMBA}")

!pip install -q /kaggle/working/{CAUSAL}
!pip install -q /kaggle/working/{MAMBA}

# 3) Patch mamba-ssm
import glob
for path in glob.glob("/usr/local/lib/python*/dist-packages/mamba_ssm/utils/generation.py"):
    with open(path) as f: src = f.read()
    new = src.replace(
        "from transformers.generation import GreedySearchDecoderOnlyOutput, SampleDecoderOnlyOutput, TextStreamer",
        "from transformers.generation import GenerateDecoderOnlyOutput, TextStreamer",
    ).replace(
        "output_cls = GreedySearchDecoderOnlyOutput if top_k == 1 else SampleDecoderOnlyOutput",
        "output_cls = GenerateDecoderOnlyOutput",
    )
    if new != src:
        with open(path, "w") as f: f.write(new)
        print(f"\u2705 Patched {path}")

print(">>> RESTART RUNTIME NOW <<<")

## 2. Repo  *(clone + cd)*

In [ ]:
# Clone the repo. The SSC vmamba.py (ssc: none|bc|abc), the build_model wiring
# in BOTH run_experiment.py and evaluate.py, the ssc configs, and the tests are
# all on main.
import os
REPO_DIR = "/kaggle/working/thesis_Choustoulakis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Rodamanthosch/thesis_Choustoulakis.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
# sanity: confirm the ssc wiring is present in both entry points
!grep -q ssc scripts/run_experiment.py && grep -q ssc scripts/evaluate.py \
    && echo "wiring OK (train + eval)" || echo "WIRING MISSING -- pull latest main"

## 3. Add `adaln_cond` support  *(idempotent patch; safe on resume)*

In [ ]:
# === Add adaln_cond / ssc_z_mlp support to the cloned repo. ===============
# Idempotent -- safe to re-run and safe to leave in on resume. Patches
# src/models/vmamba.py (7 anchored edits, backup + syntax check), wires the
# two knobs through run_experiment / evaluate / diagnose_ssc, and writes the
# two -noadaln configs. The adaln_cond=True path (every existing arm) is
# byte-identical after the patch -- regression-tested on CPU.
import base64, os
INSTALLER_B64 = """\
IyEvdXNyL2Jpbi9lbnYgYmFzaAojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09
PT09PT09PT09PQojIGluc3RhbGxfbm9hZGFsbi5zaCDigJQgYWRkIGFkYWxuX2NvbmQ9RmFsc2UgKFNTQy1PTkxZIGNvbmRp
dGlvbmluZykKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0K
IyBJbXBsZW1lbnRzIHRoZSBMSVRFUkFMIERpTS0yIHByb3Bvc2FsOiBTU0MgUkVQTEFDRVMgYWRhTE4gaW5zdGVhZCBvZgoj
IHN1cHBsZW1lbnRpbmcgaXQuIElkZW1wb3RlbnQuIFJ1biBmcm9tIHRoZSByZXBvIHJvb3QuCiMKIyBXaGF0IGFkYWxuX2Nv
bmQ9RmFsc2UgZG9lczoKIyAgICogSmlUQmxvY2s6IHRoZSBhZGFMTiBMaW5lYXIoRCAtPiA2RCkgaXMgcmVwbGFjZWQgYnkg
YSBiYXJlLCB6ZXJvLWluaXQKIyAgICAgTEVBUk5FRCBCSUFTIG9mIHNpemUgNkQuIFRoaXMgaXMgZXhhY3RseSB3aGF0IGFk
YUxOIGNvbGxhcHNlcyB0byB3aGVuCiMgICAgIGZlZCBjID0gMCAoU2lMVSgwKSA9IDAsIHNvIG9ubHkgdGhlIExpbmVhcidz
IGJpYXMgc3Vydml2ZXMpLCBpLmUuIGEKIyAgICAgcGVyLWJsb2NrIHN0YXRpYyBzaGlmdC9zY2FsZS9nYXRlIOKAlCBidXQg
d2l0aG91dCBjYXJyeWluZyAxMC45TSBkZWFkCiMgICAgIHBhcmFtZXRlcnMgdGhhdCByZWNlaXZlIG5vIGdyYWRpZW50LiBU
aGUgemVyby1pbml0IGdhdGVzIHByZXNlcnZlCiMgICAgIGFkYUxOLVplcm8ncyBpZGVudGl0eS1hdC1pbml0IGRpc2NpcGxp
bmUgZXhhY3RseS4KIyAgICogRmluYWxMYXllcjogc2FtZSB0cmVhdG1lbnQgdmlhIEZpbmFsTGF5ZXJTdGF0aWMgKGRlZmlu
ZWQgaW4gdm1hbWJhLnB5CiMgICAgIHNvIHNyYy9wcmltaXRpdmVzLnB5LCBzaGFyZWQgd2l0aCBqaXQucHkgYW5kIHZpbS5w
eSwgaXMgdW50b3VjaGVkKS4KIyAgICogKHQsIHkpIHRoZXJlZm9yZSBDQU5OT1QgcmVhY2ggdGhlIG5ldHdvcmsgdGhyb3Vn
aCB0aGUgbm9ybWFsaXphdGlvbgojICAgICBwYXRoOyB0aGUgT05MWSBjb25kaXRpb25pbmcgY2hhbm5lbCBpcyBTU0MncyBC
L0MgKCtBKSBtb2R1bGF0aW9uLgojCiMgICBzc2Nfel9tbHA9VHJ1ZSBhZGRpdGlvbmFsbHkgZ2l2ZXMgU1NDIHRoZSBkZWRp
Y2F0ZWQgeiA9IE1MUCh0LCBjKQojICAgcHJvamVjdGlvbiBEaU0tMiBzcGVjaWZpZXMsIHNvIHRoZSByZXBsYWNlbWVudCBh
cm0gaXMgbm90IGhhbmRpY2FwcGVkIGJ5CiMgICBhIGNvbmRpdGlvbiByZXByZXNlbnRhdGlvbiB0aGF0IHdhcyBzaGFwZWQg
Zm9yIGFkYUxOLiBPbmx5IGxlZ2FsIHdpdGgKIyAgIGFkYWxuX2NvbmQ9RmFsc2UgKGFzc2VydGVkKS4KIwojIEd1YXJhbnRl
ZXM6IGFkYWxuX2NvbmQ9VHJ1ZSBpcyB0aGUgREVGQVVMVCBhbmQgdGhhdCBwYXRoIGlzIGJ5dGUtaWRlbnRpY2FsCiMgdG8g
dGhlIGN1cnJlbnQgbW9kZWwgKHJlZ3Jlc3Npb24tdGVzdGVkKSwgc28gZXZlcnkgZXhpc3RpbmcgYXJtIOKAlCBiYXNlbGlu
ZSwKIyBpbi1jb250ZXh0LCBzdGF0ZS1pbml0LCBzc2Mgc3RhdGljL2JjL2FiYyDigJQgaXMgdW5hZmZlY3RlZC4KIyA9PT09
PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0Kc2V0IC1ldW8gcGlwZWZh
aWwKZXhwb3J0IFBZVEhPTlVURjg9MQoKVEFSR0VUPSJzcmMvbW9kZWxzL3ZtYW1iYS5weSIKWyAtZiAiJFRBUkdFVCIgXSB8
fCB7IGVjaG8gIkVSUk9SOiBydW4gZnJvbSB0aGUgcmVwbyByb290ICgkVEFSR0VUIG5vdCBmb3VuZCkiOyBleGl0IDE7IH0K
CmlmIGdyZXAgLXEgImFkYWxuX2NvbmQiICIkVEFSR0VUIjsgdGhlbgogIGVjaG8gInZtYW1iYS5weSBhbHJlYWR5IHN1cHBv
cnRzIGFkYWxuX2NvbmQg4oCUIHNraXBwaW5nIHBhdGNoLiIKZWxzZQogIGNwICIkVEFSR0VUIiAiJFRBUkdFVC5iYWstbm9h
ZGFsbiIKICBweXRob24gLSAiJFRBUkdFVCIgPDwnUFlFT0YnCmltcG9ydCBzeXMsIHB5X2NvbXBpbGUKClBBVEggPSBzeXMu
YXJndlsxXQpzcmMgPSBvcGVuKFBBVEgpLnJlYWQoKQoKRURJVFMgPSBbCiMg4pSA4pSAIDEuIEppVEJsb2NrIHNpZ25hdHVy
ZSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi
lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi
lIDilIDilIDilIDilIAKKCcnJyAgICBkZWYgX19pbml0X18oc2VsZiwgaGlkZGVuX3NpemUsIG51bV9oZWFkcz1Ob25lLCBt
bHBfcmF0aW89NC4wLAogICAgICAgICAgICAgICAgIGRfc3RhdGU9MTYsIGRfY29udj0zLCBleHBhbmQ9MSwgSz00LAogICAg
ICAgICAgICAgICAgIGF0dG5fZHJvcD0wLjAsIHByb2pfZHJvcD0wLjAsIHN0YXRlX2luaXQ9Im5vbmUiLCBzc2M9Im5vbmUi
KTonJycsCiAnJycgICAgZGVmIF9faW5pdF9fKHNlbGYsIGhpZGRlbl9zaXplLCBudW1faGVhZHM9Tm9uZSwgbWxwX3JhdGlv
PTQuMCwKICAgICAgICAgICAgICAgICBkX3N0YXRlPTE2LCBkX2NvbnY9MywgZXhwYW5kPTEsIEs9NCwKICAgICAgICAgICAg
ICAgICBhdHRuX2Ryb3A9MC4wLCBwcm9qX2Ryb3A9MC4wLCBzdGF0ZV9pbml0PSJub25lIiwgc3NjPSJub25lIiwKICAgICAg
ICAgICAgICAgICBhZGFsbl9jb25kOiBib29sID0gVHJ1ZSk6JycnKSwKCiMg4pSA4pSAIDIuIEppVEJsb2NrOiBjb25kaXRp
b25hbCBhZGFMTiBjb25zdHJ1Y3Rpb24g4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA
4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACignJycgICAgICAgIHNlbGYuYWRhTE5fbW9kdWxh
dGlvbiA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLlNpTFUoKSwKICAgICAgICAgICAgbm4uTGluZWFyKGhpZGRl
bl9zaXplLCA2ICogaGlkZGVuX3NpemUsIGJpYXM9VHJ1ZSksCiAgICAgICAgKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgs
IGMsIEgsIFcpOgogICAgICAgIHNoaWZ0X21zYSwgc2NhbGVfbXNhLCBnYXRlX21zYSwgc2hpZnRfbWxwLCBzY2FsZV9tbHAs
IGdhdGVfbWxwID0gXFwKICAgICAgICAgICAgc2VsZi5hZGFMTl9tb2R1bGF0aW9uKGMpLmNodW5rKDYsIGRpbT0tMSknJycs
CiAnJycgICAgICAgIHNlbGYuYWRhbG5fY29uZCA9IGFkYWxuX2NvbmQKICAgICAgICBpZiBhZGFsbl9jb25kOgogICAgICAg
ICAgICBzZWxmLmFkYUxOX21vZHVsYXRpb24gPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICAgICAgbm4uU2lMVSgpLAog
ICAgICAgICAgICAgICAgbm4uTGluZWFyKGhpZGRlbl9zaXplLCA2ICogaGlkZGVuX3NpemUsIGJpYXM9VHJ1ZSksCiAgICAg
ICAgICAgICkKICAgICAgICBlbHNlOgogICAgICAgICAgICAjIGFkYUxOIGZlZCBhIENPTlNUQU5UOiBTaUxVKDApPTAgY29s
bGFwc2VzIHRoZSBMaW5lYXIgdG8gaXRzIGJpYXMsCiAgICAgICAgICAgICMgc28gdGhpcyBiYXJlIHplcm8taW5pdCBwYXJh
bWV0ZXIgaXMgZnVuY3Rpb25hbGx5IGlkZW50aWNhbCB0bwogICAgICAgICAgICAjIGFkYUxOLXdpdGgtYz0wIHdoaWxlIGRy
b3BwaW5nIHRoZSA2RCB4IEQgZGVhZCB3ZWlnaHQgbWF0cml4LgogICAgICAgICAgICAjIFplcm8taW5pdCBrZWVwcyBnYXRl
X21zYSA9IGdhdGVfbWxwID0gMCAtPiBpZGVudGl0eSBhdCBpbml0LgogICAgICAgICAgICBzZWxmLmFkYUxOX2JpYXMgPSBu
bi5QYXJhbWV0ZXIodG9yY2guemVyb3MoNiAqIGhpZGRlbl9zaXplKSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4LCBjLCBI
LCBXKToKICAgICAgICBtb2QgPSAoc2VsZi5hZGFMTl9tb2R1bGF0aW9uKGMpIGlmIHNlbGYuYWRhbG5fY29uZAogICAgICAg
ICAgICAgICBlbHNlIHNlbGYuYWRhTE5fYmlhc1tOb25lLCA6XS5leHBhbmQoeC5zaGFwZVswXSwgLTEpKQogICAgICAgIHNo
aWZ0X21zYSwgc2NhbGVfbXNhLCBnYXRlX21zYSwgc2hpZnRfbWxwLCBzY2FsZV9tbHAsIGdhdGVfbWxwID0gXFwKICAgICAg
ICAgICAgbW9kLmNodW5rKDYsIGRpbT0tMSknJycpLAoKIyDilIDilIAgMy4gRmluYWxMYXllclN0YXRpYywgZGVmaW5lZCBu
ZXh0IHRvIEppVEJsb2NrIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU
gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAooJycnIyDilIDilIAgSmlULVZNYW1iYSBtb2RlbCAoZnJvbSBqaXQtdm1h
bWJhLWNpZmFyMTAgQ2VsbCAxNSkg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA
4pSA4pSA4pSA4pSA4pSA4pSAJycnLAogJycnY2xhc3MgRmluYWxMYXllclN0YXRpYyhubi5Nb2R1bGUpOgogICAgIiIiRmlu
YWxMYXllciB3aXRoIGNvbmRpdGlvbi1pbmRlcGVuZGVudCAoYmFyZSBiaWFzKSBtb2R1bGF0aW9uLgoKICAgIFVzZWQgd2hl
biBhZGFsbl9jb25kPUZhbHNlOiBpZGVudGljYWwgYWxnZWJyYSB0byBGaW5hbExheWVyIGZlZCBjID0gMCwKICAgIHdpdGhv
dXQgdGhlIGRlYWQgTGluZWFyKEQgLT4gMkQpLiBaZXJvLWluaXQgc2hpZnQvc2NhbGUgKyB6ZXJvLWluaXQKICAgIG91dHB1
dCBrZWVwIHRoZSBtb2RlbCdzIGluaXQgYmVoYXZpb3VyIGlkZW50aWNhbCB0byB0aGUgYmFzZWxpbmUuCiAgICAiIiIKICAg
IGRlZiBfX2luaXRfXyhzZWxmLCBoaWRkZW5fc2l6ZSwgcGF0Y2hfc2l6ZSwgb3V0X2NoYW5uZWxzKToKICAgICAgICBzdXBl
cigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLm5vcm1fZmluYWwgPSBSTVNOb3JtKGhpZGRlbl9zaXplKQogICAgICAgIHNl
bGYubGluZWFyID0gbm4uTGluZWFyKGhpZGRlbl9zaXplLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHBhdGNo
X3NpemUgKiBwYXRjaF9zaXplICogb3V0X2NoYW5uZWxzLCBiaWFzPVRydWUpCiAgICAgICAgc2VsZi5hZGFMTl9iaWFzID0g
bm4uUGFyYW1ldGVyKHRvcmNoLnplcm9zKDIgKiBoaWRkZW5fc2l6ZSkpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCwgYyk6
CiAgICAgICAgc2hpZnQsIHNjYWxlID0gc2VsZi5hZGFMTl9iaWFzW05vbmUsIDpdLmV4cGFuZCh4LnNoYXBlWzBdLCAtMSku
Y2h1bmsoMiwgZGltPTEpCiAgICAgICAgeCA9IG1vZHVsYXRlKHNlbGYubm9ybV9maW5hbCh4KSwgc2hpZnQsIHNjYWxlKQog
ICAgICAgIHJldHVybiBzZWxmLmxpbmVhcih4KQoKCiMg4pSA4pSAIEppVC1WTWFtYmEgbW9kZWwgKGZyb20gaml0LXZtYW1i
YS1jaWZhcjEwIENlbGwgMTUpIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU
gOKUgOKUgOKUgOKUgOKUgCcnJyksCgojIOKUgOKUgCA0LiBKaVRWTWFtYmEgc2lnbmF0dXJlIOKUgOKUgOKUgOKUgOKUgOKU
gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU
gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAooJycnICAg
ICAgICBzc2M6IHN0ciA9ICJub25lIiwKICAgICk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpJycnLAogJycnICAgICAg
ICBzc2M6IHN0ciA9ICJub25lIiwKICAgICAgICAjIOKUgOKUgCBTU0MgYXMgYSBSRVBMQUNFTUVOVCBmb3IgYWRhTE4gKGxp
dGVyYWwgRGlNLTIpLCBvZmYgYnkgZGVmYXVsdCDilIDilIAKICAgICAgICAjICAgYWRhbG5fY29uZD1UcnVlICDihpIgYWRh
TE4tWmVybyBjYXJyaWVzICh0LCB5KTsgZXZlcnkgZXhpc3RpbmcgYXJtLgogICAgICAgICMgICBhZGFsbl9jb25kPUZhbHNl
IOKGkiBhZGFMTiBiZWNvbWVzIGEgY29uZGl0aW9uLUlOREVQRU5ERU5UIGxlYXJuZWQKICAgICAgICAjICAgICAgICAgICAg
ICAgICAgICAgIGJpYXM7ICh0LCB5KSByZWFjaCB0aGUgbmV0d29yayBPTkxZIHRocm91Z2ggU1NDLgogICAgICAgICMgICAg
ICAgICAgICAgICAgICAgICAgUmVxdWlyZXMgc3NjICE9ICJub25lIiAoZWxzZSBub3RoaW5nIGlzCiAgICAgICAgIyAgICAg
ICAgICAgICAgICAgICAgICBjb25kaXRpb25hbCBhdCBhbGwpLiBTYXZlcyB+MTAuOU0gcGFyYW1zIGF0IFMvMTIuCiAgICAg
ICAgIyAgIHNzY196X21scD1UcnVlICAg4oaSIGdpdmUgU1NDIERpTS0yJ3MgZGVkaWNhdGVkIHogPSBNTFAodCwgYykgaW5z
dGVhZAogICAgICAgICMgICAgICAgICAgICAgICAgICAgICAgb2YgdGhlIHJhdyBzaGFyZWQgYy4gT25seSBsZWdhbCB3aXRo
IGFkYWxuIG9mZi4KICAgICAgICBhZGFsbl9jb25kOiBib29sID0gVHJ1ZSwKICAgICAgICBzc2Nfel9tbHA6IGJvb2wgPSBG
YWxzZSwKICAgICk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgYXNzZXJ0IGFkYWxuX2NvbmQgb3Igc3Nj
ICE9ICJub25lIiwgKAogICAgICAgICAgICAiYWRhbG5fY29uZD1GYWxzZSByZW1vdmVzIHRoZSBPTkxZIG90aGVyIGNvbmRp
dGlvbmluZyBwYXRoOyAiCiAgICAgICAgICAgICJlbmFibGUgYW4gc3NjIGFybSAoc3RhdGljL2JjL2FiYykgb3Iga2VlcCBh
ZGFMTiBvbi4iCiAgICAgICAgKQogICAgICAgIGFzc2VydCBhZGFsbl9jb25kIG9yIGluX2NvbnRleHRfbGVuID09IDAsICgK
ICAgICAgICAgICAgImFkYWxuX2NvbmQ9RmFsc2UgaXMgdGhlIFNTQy1yZXBsYWNlbWVudCBhcm07IGNvbWJpbmluZyBpdCB3
aXRoICIKICAgICAgICAgICAgInRoZSBpbi1jb250ZXh0IHByZWZpeCBtaXhlcyB0d28gc2VwYXJhdGUgZXhwZXJpbWVudHMu
IgogICAgICAgICkKICAgICAgICBhc3NlcnQgbm90IHNzY196X21scCBvciBub3QgYWRhbG5fY29uZCwgKAogICAgICAgICAg
ICAic3NjX3pfbWxwIHJld2lyZXMgdGhlIHZlY3RvciBhZGFMTiBjb25zdW1lczsgb25seSB1c2UgaXQgd2l0aCAiCiAgICAg
ICAgICAgICJhZGFsbl9jb25kPUZhbHNlLiIKICAgICAgICApCiAgICAgICAgc2VsZi5hZGFsbl9jb25kID0gYWRhbG5fY29u
ZAogICAgICAgIHNlbGYuc3NjX3pfbWxwID0gc3NjX3pfbWxwJycnKSwKCiMg4pSA4pSAIDUuIGJsb2NrIGNvbnN0cnVjdGlv
biArIGZpbmFsIGxheWVyICsgei1NTFAg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA
4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACignJycgICAgICAgICAgICAgICAgc3RhdGVfaW5p
dD1zdGF0ZV9pbml0LCBzc2M9c3NjLAogICAgICAgICAgICApCiAgICAgICAgICAgIGZvciBpIGluIHJhbmdlKGRlcHRoKQog
ICAgICAgIF0pCgogICAgICAgIHNlbGYuZmluYWxfbGF5ZXIgPSBGaW5hbExheWVyKGhpZGRlbl9zaXplLCBwYXRjaF9zaXpl
LCBzZWxmLm91dF9jaGFubmVscyknJycsCiAnJycgICAgICAgICAgICAgICAgc3RhdGVfaW5pdD1zdGF0ZV9pbml0LCBzc2M9
c3NjLCBhZGFsbl9jb25kPWFkYWxuX2NvbmQsCiAgICAgICAgICAgICkKICAgICAgICAgICAgZm9yIGkgaW4gcmFuZ2UoZGVw
dGgpCiAgICAgICAgXSkKCiAgICAgICAgIyBEaU0tMidzIHogPSBNTFAodCwgYyk7IG9ubHkgYnVpbHQgZm9yIHRoZSBhZGFM
Ti1yZXBsYWNlbWVudCBhcm0uCiAgICAgICAgc2VsZi5zc2NfeiA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkxp
bmVhcihoaWRkZW5fc2l6ZSwgaGlkZGVuX3NpemUpLAogICAgICAgICAgICBubi5TaUxVKCksCiAgICAgICAgICAgIG5uLkxp
bmVhcihoaWRkZW5fc2l6ZSwgaGlkZGVuX3NpemUpLAogICAgICAgICkgaWYgc3NjX3pfbWxwIGVsc2UgTm9uZQoKICAgICAg
ICBzZWxmLmZpbmFsX2xheWVyID0gKEZpbmFsTGF5ZXIgaWYgYWRhbG5fY29uZCBlbHNlIEZpbmFsTGF5ZXJTdGF0aWMpKAog
ICAgICAgICAgICBoaWRkZW5fc2l6ZSwgcGF0Y2hfc2l6ZSwgc2VsZi5vdXRfY2hhbm5lbHMpJycnKSwKCiMg4pSA4pSAIDYu
IGluaXRpYWxpemVfd2VpZ2h0czogZ3VhcmQgdGhlIGFkYUxOLVplcm8gYmxvY2sg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA
4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACignJycgICAgICAgICMgYWRhTE4tWmVy
bwogICAgICAgIGZvciBibG9jayBpbiBzZWxmLmJsb2NrczoKICAgICAgICAgICAgbm4uaW5pdC5jb25zdGFudF8oYmxvY2su
YWRhTE5fbW9kdWxhdGlvblstMV0ud2VpZ2h0LCAwKQogICAgICAgICAgICBubi5pbml0LmNvbnN0YW50XyhibG9jay5hZGFM
Tl9tb2R1bGF0aW9uWy0xXS5iaWFzLCAwKQogICAgICAgIG5uLmluaXQuY29uc3RhbnRfKHNlbGYuZmluYWxfbGF5ZXIuYWRh
TE5fbW9kdWxhdGlvblstMV0ud2VpZ2h0LCAwKQogICAgICAgIG5uLmluaXQuY29uc3RhbnRfKHNlbGYuZmluYWxfbGF5ZXIu
YWRhTE5fbW9kdWxhdGlvblstMV0uYmlhcywgMCknJycsCiAnJycgICAgICAgICMgYWRhTE4tWmVybyAob3IsIHdpdGggYWRh
bG5fY29uZD1GYWxzZSwgdGhlIHplcm8taW5pdCBzdGF0aWMgYmlhc2VzCiAgICAgICAgIyB0aGF0IHJlcGxhY2UgaXQg4oCU
IHNhbWUgaWRlbnRpdHktYXQtaW5pdCBndWFyYW50ZWUgZWl0aGVyIHdheSkuCiAgICAgICAgaWYgc2VsZi5hZGFsbl9jb25k
OgogICAgICAgICAgICBmb3IgYmxvY2sgaW4gc2VsZi5ibG9ja3M6CiAgICAgICAgICAgICAgICBubi5pbml0LmNvbnN0YW50
XyhibG9jay5hZGFMTl9tb2R1bGF0aW9uWy0xXS53ZWlnaHQsIDApCiAgICAgICAgICAgICAgICBubi5pbml0LmNvbnN0YW50
XyhibG9jay5hZGFMTl9tb2R1bGF0aW9uWy0xXS5iaWFzLCAwKQogICAgICAgICAgICBubi5pbml0LmNvbnN0YW50XyhzZWxm
LmZpbmFsX2xheWVyLmFkYUxOX21vZHVsYXRpb25bLTFdLndlaWdodCwgMCkKICAgICAgICAgICAgbm4uaW5pdC5jb25zdGFu
dF8oc2VsZi5maW5hbF9sYXllci5hZGFMTl9tb2R1bGF0aW9uWy0xXS5iaWFzLCAwKQogICAgICAgIGVsc2U6CiAgICAgICAg
ICAgIGZvciBibG9jayBpbiBzZWxmLmJsb2NrczoKICAgICAgICAgICAgICAgIG5uLmluaXQuY29uc3RhbnRfKGJsb2NrLmFk
YUxOX2JpYXMsIDApCiAgICAgICAgICAgIG5uLmluaXQuY29uc3RhbnRfKHNlbGYuZmluYWxfbGF5ZXIuYWRhTE5fYmlhcywg
MCknJycpLAoKIyDilIDilIAgNy4gZm9yd2FyZDogcm91dGUgeiB0aHJvdWdoIHRoZSBTU0MgcGF0aCDilIDilIDilIDilIDi
lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi
lIDilIDilIDilIDilIAKKCcnJyAgICAgICAgICAgIHggPSBibG9jayh4LCBjLCBILCBXKScnJywKICcnJyAgICAgICAgICAg
ICMgV2l0aCBzc2Nfel9tbHAgdGhlIGJsb2NrcyByZWNlaXZlIERpTS0yJ3MgeiBpbnN0ZWFkIG9mIHRoZSByYXcKICAgICAg
ICAgICAgIyBjLiBMZWdhbCBvbmx5IHdoZW4gYWRhbG5fY29uZD1GYWxzZSwgd2hlcmUgdGhlIGJsb2NrIGNvbnN1bWVzCiAg
ICAgICAgICAgICMgdGhpcyB2ZWN0b3Igc29sZWx5IGFzIHRoZSBTU0MgY29uZGl0aW9uIChhZGFMTiBpZ25vcmVzIGl0KS4K
ICAgICAgICAgICAgeCA9IGJsb2NrKHgsIHNlbGYuc3NjX3ooYykgaWYgc2VsZi5zc2NfeiBpcyBub3QgTm9uZSBlbHNlIGMs
IEgsIFcpJycnKSwKXQoKZm9yIG9sZCwgbmV3IGluIEVESVRTOgogICAgYXNzZXJ0IHNyYy5jb3VudChvbGQpID09IDEsICJh
bmNob3Igbm90IGZvdW5kL3VuaXF1ZTpcbiIgKyBvbGRbOjEyMF0KICAgIHNyYyA9IHNyYy5yZXBsYWNlKG9sZCwgbmV3KQoK
b3BlbihQQVRILCAidyIpLndyaXRlKHNyYykKcHlfY29tcGlsZS5jb21waWxlKFBBVEgsIGRvcmFpc2U9VHJ1ZSkKcHJpbnQo
ZiJQYXRjaGVkIHtQQVRIfTogYWRhbG5fY29uZCAvIHNzY196X21scCBhZGRlZCAoNyBlZGl0cyksIHN5bnRheCBPSy4iKQpQ
WUVPRgogIGlmIFsgJD8gLW5lIDAgXTsgdGhlbgogICAgZWNobyAiUEFUQ0ggRkFJTEVEIOKAlCByZXN0b3JpbmcgYmFja3Vw
IjsgY3AgIiRUQVJHRVQuYmFrLW5vYWRhbG4iICIkVEFSR0VUIjsgZXhpdCAxCiAgZmkKZmkKCiMg4pSA4pSAIHdpcmUgdGhl
IHR3byBrbm9icyB0aHJvdWdoIHRoZSBlbnRyeSBwb2ludHMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA
4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACmZvciBGIGluIHNjcmlwdHMvcnVu
X2V4cGVyaW1lbnQucHkgc2NyaXB0cy9ldmFsdWF0ZS5weSBzY3JpcHRzL2RpYWdub3NlX3NzYy5weTsgZG8KICBbIC1mICIk
RiIgXSB8fCBjb250aW51ZQogIGlmIGdyZXAgLXEgImFkYWxuX2NvbmQiICIkRiI7IHRoZW4KICAgIGVjaG8gIiRGIGFscmVh
ZHkgd2lyZWQuIgogIGVsc2UKICAgIHB5dGhvbiAtICIkRiIgPDwnUFlFT0YnCmltcG9ydCBzeXMKUEFUSCA9IHN5cy5hcmd2
WzFdCnMgPSBvcGVuKFBBVEgpLnJlYWQoKQpvbGQgPSAnICAgICAgICBzc2M9bS5nZXQoInNzYyIsICJub25lIiksXG4nCm5l
dyA9ICgnICAgICAgICBzc2M9bS5nZXQoInNzYyIsICJub25lIiksXG4nCiAgICAgICAnICAgICAgICBhZGFsbl9jb25kPW0u
Z2V0KCJhZGFsbl9jb25kIiwgVHJ1ZSksXG4nCiAgICAgICAnICAgICAgICBzc2Nfel9tbHA9bS5nZXQoInNzY196X21scCIs
IEZhbHNlKSxcbicpCm9sZDIgPSAnICAgICAgICBzc2M9bV9jZmcuZ2V0KCJzc2MiLCAibm9uZSIpLFxuJwpuZXcyID0gKCcg
ICAgICAgIHNzYz1tX2NmZy5nZXQoInNzYyIsICJub25lIiksXG4nCiAgICAgICAgJyAgICAgICAgYWRhbG5fY29uZD1tX2Nm
Zy5nZXQoImFkYWxuX2NvbmQiLCBUcnVlKSxcbicKICAgICAgICAnICAgICAgICBzc2Nfel9tbHA9bV9jZmcuZ2V0KCJzc2Nf
el9tbHAiLCBGYWxzZSksXG4nKQppZiBzLmNvdW50KG9sZCkgPT0gMToKICAgIHMgPSBzLnJlcGxhY2Uob2xkLCBuZXcpCmVs
aWYgcy5jb3VudChvbGQyKSA9PSAxOgogICAgcyA9IHMucmVwbGFjZShvbGQyLCBuZXcyKQplbHNlOgogICAgcHJpbnQoZiIg
IChubyBidWlsZF9tb2RlbCBhbmNob3IgaW4ge1BBVEh9OyBza2lwcGVkKSIpOyByYWlzZSBTeXN0ZW1FeGl0KDApCm9wZW4o
UEFUSCwgInciKS53cml0ZShzKQpwcmludChmIiAgd2lyZWQge1BBVEh9IikKUFlFT0YKICBmaQpkb25lCgojIOKUgOKUgCBj
b25maWdzIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU
gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU
gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgApmb3IgQVJNIGluIGJj
IGFiYzsgZG8KICBTUkM9ImNvbmZpZ3MvdGlueV9pbWFnZW5ldC9qaXQtczItdm1hbWJhLXNzYy0ke0FSTX0ueWFtbCIKICBE
U1Q9ImNvbmZpZ3MvdGlueV9pbWFnZW5ldC9qaXQtczItdm1hbWJhLXNzYy0ke0FSTX0tbm9hZGFsbi55YW1sIgogIGlmIFsg
LWYgIiREU1QiIF07IHRoZW4KICAgIGVjaG8gIiREU1QgYWxyZWFkeSBleGlzdHMg4oCUIGxlZnQgdW50b3VjaGVkLiIKICBl
bGlmIFsgLWYgIiRTUkMiIF07IHRoZW4KICAgIHB5dGhvbiAtICIkU1JDIiAiJERTVCIgIiRBUk0iIDw8J1BZRU9GJwppbXBv
cnQgc3lzCnNyYywgZHN0LCBhcm0gPSBzeXMuYXJndlsxXSwgc3lzLmFyZ3ZbMl0sIHN5cy5hcmd2WzNdCnMgPSBvcGVuKHNy
YykucmVhZCgpCnMgPSBzLnJlcGxhY2UoZiJuYW1lOiBqaXQtczItdm1hbWJhLXRpbnlpbi1zc2Mte2FybX0iLAogICAgICAg
ICAgICAgIGYibmFtZTogaml0LXMyLXZtYW1iYS10aW55aW4tc3NjLXthcm19LW5vYWRhbG4iKQpzID0gcy5yZXBsYWNlKGYi
b3V0cHV0X2RpcjogZXhwZXJpbWVudHMvdGlueV9pbWFnZW5ldC9qaXQtczItdm1hbWJhLXNzYy17YXJtfSIsCiAgICAgICAg
ICAgICAgZiJvdXRwdXRfZGlyOiBleHBlcmltZW50cy90aW55X2ltYWdlbmV0L2ppdC1zMi12bWFtYmEtc3NjLXthcm19LW5v
YWRhbG4iKQpzID0gcy5yZXBsYWNlKGYiICBzc2M6IHthcm19IiwgZiIiIiAgc3NjOiB7YXJtfQogICMgLS0gU1NDIFJFUExB
Q0VTIGFkYUxOIChsaXRlcmFsIERpTS0yKSwgcmF0aGVyIHRoYW4gc3VwcGxlbWVudGluZyBpdCAtLQogICMgYWRhTE4gYmVj
b21lcyBhIGNvbmRpdGlvbi1pbmRlcGVuZGVudCBsZWFybmVkIGJpYXM6ICh0LCB5KSByZWFjaCB0aGUKICAjIG5ldHdvcmsg
T05MWSB0aHJvdWdoIFNTQydzIEIvQygrQSkgbW9kdWxhdGlvbi4gfjIwLjdNIHBhcmFtcyB2cyAzMS42TQogICMgKGFkYUxO
IGlzIDM1JSBvZiB0aGUgbW9kZWwpLCBidXQgb25seSB+MSUgbGVzcyBjb21wdXRlIC0tIGFkYUxOIHJ1bnMKICAjIG9uY2Ug
cGVyIFNBTVBMRSwgbm90IHBlciB0b2tlbi4KICBhZGFsbl9jb25kOiBmYWxzZQogIHNzY196X21scDogdHJ1ZSAgICAgICAg
ICAgICAgICAgICAjIERpTS0yJ3MgZGVkaWNhdGVkIHogPSBNTFAodCwgYykiIiIpCm9wZW4oZHN0LCAidyIpLndyaXRlKHMp
CnByaW50KGYiV3JvdGUge2RzdH0iKQpQWUVPRgogIGVsc2UKICAgIGVjaG8gIldBUk5JTkc6ICRTUkMgbm90IGZvdW5kOyBz
a2lwcGluZyAkRFNUIgogIGZpCmRvbmUKCmVjaG8gImluc3RhbGxfbm9hZGFsbi5zaCDigJQgRE9ORS4iCg=="""
open("install_noadaln.sh", "wb").write(base64.b64decode(INSTALLER_B64.replace("\n", "")))
!bash install_noadaln.sh

## 4. Verify the arm  *(first session only; ~1 min)*

In [ ]:
# === Verify the SSC arm (~1 min; first session only -- comment out on resume).
# 1) equivalence proofs: gate identity exp((s*Delta)A)=exp(Delta*A*s) with B/s,
#    s=1 consistency, bias linearity -- machine epsilon vs selective_scan_ref
# 2b) no-adaLN guards: identity-at-init, (t,y) reach the net ONLY via
#     SSC, param accounting, and adaln_cond=True byte-identity.
# 2) model guards: bit-identity of ssc=none / init semantics / grads / gate /
#    arm exclusivity
# PYTHONPATH=. is REQUIRED: without it "from src.models.vmamba import" fails
# and the tests' try/except misreports it as a missing mamba_ssm (known issue).
# If (1) FAILS after a Kaggle torch/CUDA bump, STOP -- do not train on an
# untrusted kernel build.
!PYTHONPATH=. python tests/test_ssc_equivalence.py
!PYTHONPATH=. python tests/test_ssc_model.py
!PYTHONPATH=. python tests/test_noadaln_cpu.py

## 5. Download Tiny-ImageNet  *(to /tmp; comment out if you attach a dataset)*

In [ ]:
# Download Tiny-ImageNet-200 to /tmp (EPHEMERAL: keeps your saved Version small --
# only checkpoints go to /kaggle/working). Re-downloads each session (~2-3 min).
# Faster alternative: attach a Kaggle tiny-imagenet dataset and set DATA_DIR to it,
# then comment this cell out. Requires Internet ON.
import os, zipfile, glob, shutil
DATA_DIR = "/tmp/tiny-imagenet-200"
ZIP      = "/tmp/tiny-imagenet-200.zip"
SRC      = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"

if not os.path.isdir(os.path.join(DATA_DIR, "train")):
    if not os.path.exists(ZIP):
        print("downloading tiny-imagenet-200 (~240MB)...")
        os.system(f"wget -q -O {ZIP} {SRC}")
    print("extracting...")
    with zipfile.ZipFile(ZIP) as z:
        z.extractall("/tmp")

# TRAIN: train/<cls>/images/*.JPEG -> train/<cls>/*.JPEG  (ImageFolder layout)
tr = os.path.join(DATA_DIR, "train")
for cls in os.listdir(tr):
    sub = os.path.join(tr, cls, "images")
    if os.path.isdir(sub):
        for f in glob.glob(os.path.join(sub, "*.JPEG")):
            shutil.move(f, os.path.join(tr, cls))
        shutil.rmtree(sub)
    for b in glob.glob(os.path.join(tr, cls, "*_boxes.txt")):
        os.remove(b)

# VAL: flat val/images + val_annotations.txt -> val/<cls>/*.JPEG  (evaluate.py reads val/)
val = os.path.join(DATA_DIR, "val")
ann = os.path.join(val, "val_annotations.txt")
if os.path.exists(ann):
    with open(ann) as f:
        for line in f:
            img, cls = line.split("\t")[:2]
            os.makedirs(os.path.join(val, cls), exist_ok=True)
            s = os.path.join(val, "images", img)
            if os.path.exists(s):
                shutil.move(s, os.path.join(val, cls, img))
    shutil.rmtree(os.path.join(val, "images"), ignore_errors=True)
    os.remove(ann)

print("train classes:", len(glob.glob(tr+"/*")),
      "| val classes:", len(glob.glob(val+"/*")), "| DATA_DIR:", DATA_DIR)

## 6. Settings  *(edits the cloned config so train & eval agree)*

In [ ]:
# ── Settings for THIS notebook ─────────────────────────────────────────
SSC        = "abc"       # this notebook: abc WITHOUT adaLN (SSC-only)
EPOCHS     = 100          # full target; resume across sessions until reached
SAVE_FREQ  = 1            # overwrite checkpoint-last.pt EVERY epoch (12h safety)

# Fallback if the download cell was commented out (attached-dataset workflow):
try:
    DATA_DIR
except NameError:
    DATA_DIR = "/kaggle/input/tiny-imagenet/tiny-imagenet-200"  # <-- your attached dataset
    print("DATA_DIR not set by a download cell; using:", DATA_DIR)

CONFIG  = f"configs/tiny_imagenet/jit-s2-vmamba-ssc-{SSC}-noadaln.yaml"
OUT_DIR = "/kaggle/working/exp_ssc_abc_noadaln"   # persists in the saved Version

# Sync the cloned config so BOTH training and evaluation read identical values
# (evaluate.py has no model CLI overrides -- it builds the model straight from
# config). in_context_len stays 0 and state_init stays none: the arms are
# mutually exclusive (assert in the model).
import re
def set_cfg(path, kv):
    s = open(path).read()
    for k, v in kv.items():
        s = re.sub(rf"(?m)^(\s*{k}:).*$", rf"\1 {v}", s, count=1)
    open(path, "w").write(s)
set_cfg(CONFIG, {"data_dir": DATA_DIR, "ssc": SSC, "in_context_len": 0,
                 "state_init": "none", "adaln_cond": "false", "ssc_z_mlp": "true"})

# ── Resume across 12h sessions ─────────────────────────────────────
# 1st run: RESUME_INPUT = None. Next session: Save Version, attach THIS notebook's
# previous output as an input, set RESUME_INPUT to ".../exp_ssc_bc".
RESUME_INPUT = None
import os
RESUME_CKPT = None
for c in ([os.path.join(RESUME_INPUT, "checkpoint-last.pt")] if RESUME_INPUT else []) + \
         [os.path.join(OUT_DIR, "checkpoint-last.pt")]:
    if c and os.path.exists(c):
        RESUME_CKPT = c; break
print("config :", CONFIG, "| ssc:", SSC, "| adaLN: OFF (SSC-only)")
print("data   :", DATA_DIR, "(exists:", os.path.isdir(DATA_DIR), ")")
print("out    :", OUT_DIR, "| resume:", RESUME_CKPT or "(fresh)")

## 7. Train  *(Save Version before 12h; set RESUME_INPUT next session)*

In [ ]:
# ── TRAIN ──────────────────────────────────────────────────────────────
# data_dir + ssc now live in the (edited) config; we only override
# run-specific knobs here. Runs in a subprocess (fresh torch, no restart).
#CMD = (f"python scripts/run_experiment.py --config {CONFIG} "
#       f"checkpoint.output_dir={OUT_DIR} checkpoint.save_last_freq={SAVE_FREQ} "
 #      f"training.epochs={EPOCHS}")
#if RESUME_CKPT:
#    CMD += f" checkpoint.resume_from={RESUME_CKPT}"
#print(CMD, "\n" + "="*70)
#!{CMD}

## 8. Evaluate  *(comment out the whole cell to skip)*

In [ ]:
import os, glob
hits = sorted(glob.glob("/kaggle/input/notebooks/rodamanthos2/notebook12aeff2bca/exp_ssc_abc_noadaln/checkpoint-best.pt", recursive=True)
              or glob.glob("/kaggle/input/notebooks/rodamanthos2/notebook12aeff2bca/exp_ssc_abc_noadaln/checkpoint-last.pt", recursive=True))
EVAL_CKPT = hits[-1]
EVAL_OUT  = "/kaggle/working/exp_ssc_abc_noadaln/eval"
print("eval ckpt:", EVAL_CKPT)

EVAL_CMD = (f"python scripts/evaluate.py --config {CONFIG} --checkpoint {EVAL_CKPT} "
            f"--n_samples 10000 --batch_size 200 --ema 1 "
            f"--cfg_scale 2.5 --cfg_interval 0.1 1.0 --out_dir {EVAL_OUT}")
print(EVAL_CMD, "\n" + "="*70)
!PYTHONPATH=. {EVAL_CMD}

## Notes\n- This is the **replacement** reading of DiM-2 (SSC instead of adaLN); the `ssc-abc` arm without `-noadaln` is the **supplement** reading. Report both.\n- Params drop ~33.6% (adaLN is 35.3% of the model) while compute drops ~1% \u2014 adaLN is per-sample, not per-token. The FID comparison against the 31.03M baseline is therefore param-mismatched; say so explicitly.\n- adaLN is replaced by a zero-init learned bias, NOT deleted: the identity-at-init discipline is preserved exactly (verified: fresh model outputs 0), so only the conditioning changes, not the optimisation scaffold.\n- `ssc_z_mlp: true` gives SSC DiM-2's own `z = MLP(t, c)` so the arm is not handicapped by a condition representation shaped for adaLN.\n- Keep `EPOCHS`, `cfg_scale`, `DATA_DIR`, seed and sampler identical to every other arm.